# ANKIT LLM v0.1 — **one-click** Colab T4 run (autopilot)

Tumhe kuch configure nahi karna. **Do cell** chalao (ya sirf dusra):

1. **Cell 1** — GPU check (T4 hona chahiye: *Runtime → Change runtime type → T4 GPU*).
   Optional — Cell 2 khud bhi GPU + Drive check karta hai.
2. **Cell 2** — ye ek hi cell sab karta hai: newest `main` code sync, Drive mount +
   **DRIVE ACCOUNT print**, phir **60 minute ka chunk**. Khud dekhta hai Drive par
   checkpoint hai ya nahi (nahi → poora pipeline, hai → seedha `--mode resume`).
   Chunk khatam hone par checkpoint Drive par chala jata hai aur cell ruk jata hai —
   phir yahi cell dobara chalao, training usi step se aage badhegi.

Ye cell **3 line ki launcher** hai: asli logic `notebooks/PASTE_ONE_CELL.py` me rehta hai,
jo har run par GitHub `main` se fresh aata hai. Isliye purani notebook copy bhi kabhi
stale code nahi chalayegi (2026-11-06 ko ek teen-din purani copy ne 1 ghante tak purana
build chalaya tha — isi liye ye badla).

**Checkpoints ek rolling `latest.pt` par jaate hain**, background thread se — training
Drive ka wait nahi karti, Drive ka trash nahi badhta, aur `heartbeat.json` har 100 steps
par batata hai training kahan tak pahunchi. `reports/build_stamp.json` (local + Drive) me
git sha hota hai, taaki pata chale kaun sa build chala.

Target: **64,000 steps = ~524M tokens = ~21.8 epochs** (corpus 24.1M tokens hai — 2B token
ka matlab 83 epochs hota, jo ratne jaisa hai). Measured 1.60 steps/s ⇒ 60 min ≈ ~5,600
steps ⇒ **~11 chunks baaki** (step 5,800 se shuru).


In [ ]:
!nvidia-smi
import torch, platform
print("py", platform.python_version(), "| torch", torch.__version__)
assert torch.cuda.is_available(), "GPU nahi mila: Runtime -> Change runtime type -> T4 GPU"
p = torch.cuda.get_device_properties(0); cap = torch.cuda.get_device_capability(0)
print("GPU:", p.name, "| cc", f"{cap[0]}.{cap[1]}", "| VRAM", round(p.total_memory/1e9,1), "GB")
print("bf16 accelerated:", cap[0] >= 8, "(T4 -> False, isliye fp16 + GradScaler use hoga)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, os
DRIVE = "/content/drive/MyDrive/ankit-llm"
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE}/reports", exist_ok=True)

# ---- SABSE ZAROORI CHECK: kaun sa Google account? ---------------------------
# Drive mount jo account use karta hai, wahi checkpoint ka ghar hai. Agar Colab me
# koi doosra account logged in hai to training chalti rahegi aur checkpoints *us*
# account ki Drive me jayenge - yahan kuch nahi dikhega. Isliye account print karte
# hain aur Drive me WHOAMI.txt likh dete hain (is file se pata chalta hai kaun chala).
def drive_identity():
    try:
        from googleapiclient.discovery import build
        return build("drive", "v3").about().get(fields="user").execute().get("user", {})
    except Exception as exc:
        return {"error": f"{type(exc).__name__}: {exc}"}

who = drive_identity()
print("=" * 70)
print("DRIVE ACCOUNT:", who.get("emailAddress"), "|", who.get("displayName"))
if not who.get("emailAddress"):
    print("  account pata nahi chala ->", who.get("error"))
print("My Drive top level:", sorted(os.listdir("/content/drive/MyDrive"))[:12])
ck = f"{DRIVE}/checkpoints/t4-full"
print("checkpoints/t4-full:", sorted(os.listdir(ck))[:12] if os.path.isdir(ck) else "MISSING")
print("=" * 70)
with open(f"{DRIVE}/WHOAMI.txt", "w", encoding="utf-8") as fh:
    fh.write(json.dumps(who, indent=2) + "\n")
print("drive ready:", DRIVE)

In [ ]:
# ⬇️ Yahi ek cell chalao (Run ▶). Ye hamesha GitHub main ka newest code chalayega.
!git clone -q --depth 1 https://github.com/pushparaj9749/tool.git /content/ankit-llm 2>/dev/null || \
  (git -C /content/ankit-llm fetch -q --depth 1 origin main && git -C /content/ankit-llm reset -q --hard origin/main)
%cd /content/ankit-llm
!python notebooks/PASTE_ONE_CELL.py --budget-min 60


## Result dekho

`reports/autopilot_summary.md` (Drive me bhi copy hota hai) me sab measured numbers honge:
benchmark table, auto-selected config, pilot verification, checkpoints, mirror stats.
`reports/build_stamp.json` batata hai kaun sa git sha chala.

**Yahi files mujhe bhej dena** — usme sab kuch hoga.

## Chat try karo (GPU par fast hai)

```python
from inference.conversation import ChatEngine
eng = ChatEngine.from_checkpoint("checkpoints/t4-sft/final.pt", device="cuda")
print(eng.chat_once([{"role": "user", "content": "नमस्ते अंकित"}], mode="adaptive"))
```

## Agli session (training continue)

Wahi cell dobara chalao. Cell khud `--mode resume` chunta hai: newest checkpoint
Drive se local disk par restore hota hai, phir `--resume auto` usi LR schedule par aage
badhta hai. Sirf pichhle checkpoint ke baad ke kuch minute jaate hain.
